# Job Ads Classifier 0.2.0 Colab Smoke Test

This notebook checks the modernized `0.2.0` line on Colab.

It runs:
- repository clone
- dependency install
- CLI smoke test
- fast pytest suite
- linear smoke test
- optional transformer smoke test
- optional existing-model CPU compatibility test


In [ ]:
!rm -rf job-ads-classifier
!git clone https://github.com/OJALAB/job-ads-classifier.git
%cd job-ads-classifier
!python -m pip install --upgrade pip
!pip install -r requirements-colab-0.2.0.txt


In [ ]:
!python main.py --help
!pytest -m "not integration"


In [ ]:
!pytest tests/test_smoke_linear.py -q


## Optional transformer smoke test

This downloads a small Hugging Face model and runs a minimal CPU smoke test.


In [ ]:
import os

os.environ["RUN_TRANSFORMER_SMOKE"] = "1"
os.environ["TRANSFORMER_SMOKE_MODEL"] = "google/bert_uncased_L-2_H-128_A-2"
!pytest tests/test_smoke_transformer.py -q


## Optional existing pretrained model CPU test

Use this if you have already downloaded and extracted a pretrained model directory,
for example from RepOD or from your Google Drive.


In [ ]:
from pathlib import Path
import os

# Edit these paths before running.
EXISTING_MODEL_CLASSIFIER = "TransformerJobOffersClassifier"
EXISTING_MODEL_DIR = "/content/existing-model"
EXISTING_MODEL_INPUT = "/content/job-ads-classifier/tests/data/x_test.txt"

assert Path(EXISTING_MODEL_DIR).exists(), f"Missing model dir: {EXISTING_MODEL_DIR}"
assert Path(EXISTING_MODEL_INPUT).exists(), f"Missing input file: {EXISTING_MODEL_INPUT}"

os.environ["EXISTING_MODEL_CLASSIFIER"] = EXISTING_MODEL_CLASSIFIER
os.environ["EXISTING_MODEL_DIR"] = EXISTING_MODEL_DIR
os.environ["EXISTING_MODEL_INPUT"] = EXISTING_MODEL_INPUT

!pytest tests/test_existing_model_cpu.py -q


In [ ]:
# Optional direct prediction after the existing-model test.
# Adjust the classifier and model path if needed.
!python main.py predict TransformerJobOffersClassifier -x tests/data/x_test.txt -m /content/existing-model -p /content/predictions.txt -A cpu -P 32 -T 1
!head -n 5 /content/predictions.txt
!head -n 5 /content/predictions.txt.map
